In [3]:
import pymc as pm
import pandas as pd
import numpy as np
import arviz as az

df = pd.read_csv('full_data.csv', keep_default_na=False, na_values=[''])
df

,name,yearEstablished,state,stateAbbreviation,locationType,locationTypeKind,typeOwned,typeProfit,level,hasUndergraduate,...,stateTwoYearNetTuitionRevenuePerFTEMedian,stateFourYearNetTuitionRevenuePerFTEMedian,statePopulation18to24yoMedian,statePopulation25yoAndOverMedian,statePopulation18to24yoHighSchoolGradMedian,statePopulation25yoAndOverHighSchoolGradMedian,statePopulation18to24yoSomeCollegeOrHigherMedian,statePopulation25yoAndOverSomeCollegeOrHigherMedian,statePerCapitaPersonalIncomeMedian,reasonClosure
0,Alderson Broaddus University,1871,West Virginia,WV,Rural,Distant,Private,Nonprofit,4-year,True,...,3552.459646,8324.661159,79926.5,612398.0,60470.5,512687.5,79926.5,612398.0,51075,0
1,Alliance University (Formerly Nyack College),1882,New York,NY,City,Large,Private,Nonprofit,4-year,True,...,4968.842112,7115.076497,1126667.0,8594351.5,497059.0,3517833.0,1126667.0,8594351.5,78501,0
2,Ancilla College,1937,Indiana,IN,Rural,Distant,Private,Nonprofit,2-year,True,...,3082.258414,12604.847801,337715.5,2531049.0,228011.0,1484195.5,337715.5,2531049.0,59797,3
3,Becker College,1784,Massachusetts,MA,City,Midsize,Private,Nonprofit,4-year,True,...,3823.431201,8164.798013,433897.0,3292373.5,194673.5,1135221.5,433897.0,3292373.5,85585,12
4,Bloomfield College,1868,New Jersey,NJ,Suburb,Large,Private,Nonprofit,4-year,True,...,3230.813658,11047.054665,469274.0,4014028.0,225395.0,1672783.5,469274.0,4014028.0,78329,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,Eastern Nazarene College,1900,Massachusetts,MA,Suburb,Large,Private,Nonprofit,4-year,True,...,3823.431201,8164.798013,433897.0,3292373.5,194673.5,1135221.5,433897.0,3292373.5,85585,0
61,Cox College,1907,Missouri,MO,City,Midsize,Private,Nonprofit,4-year,True,...,3158.651054,8110.752686,318494.5,2520719.5,185163.0,1276294.0,318494.5,2520719.5,59202,0
62,University of Wisconsin–Platteville Richland,1967,Wisconsin,WI,Town,Distant,Public,Nonprofit,4-year,True,...,2499.698379,9046.426414,313717.5,2507124.5,179977.5,1210502.5,313717.5,2507124.5,62713,1
63,Maryland University of Integrative Health,1974,Maryland,MD,Suburb,Large,Private,Nonprofit,graduate school,False,...,4758.870615,11787.159727,306338.5,2800406.0,166310.5,1014233.5,306338.5,2800406.0,71897,3


## Adding data

In [4]:
stateAbbri = {"Alabama": "AL",
             "Alaska": "AK",
             "Arizona": "AZ",
             "Arkansas": "AR",
             "California": "CA",
             "Colorado": "CO",
             "Connecticut": "CT",
             "Delaware": "DE",
             "Florida": "FL",
             "Georgia": "GA",
             "Hawaii": "HI",
             "Idaho": "ID",
             "Illinois": "IL",
             "Indiana": "IN",
             "Iowa": "IA",
             "Kansas": "KS",
             "Kentucky": "KY",
             "Louisiana": "LA",
             "Maine": "ME",
             "Maryland": "MD",
             "Massachusetts": "MA",
             "Michigan": "MI",
             "Minnesota": "MN",
             "Mississippi": "MS",
             "Missouri": "MO",
             "Montana": "MT",
             "Nebraska": "NE",
             "Nevada": "NV",
             "New Hampshire": "NH",
             "New Jersey": "NJ",
             "New Mexico": "NM",
             "New York": "NY",
             "North Carolina": "NC",
             "North Dakota": "ND",
             "Ohio": "OH",
             "Oklahoma": "OK",
             "Oregon": "OR",
             "Pennsylvania": "PA",
             "Rhode Island": "RI",
             "South Carolina": "SC",
             "South Dakota": "SD",
             "Tennessee": "TN",
             "Texas": "TX",
             "Utah": "UT",
             "Vermont": "VT",
             "Virginia": "VA",
             "Washington": "WA",
             "West Virginia": "WV",
             "Wisconsin": "WI",
             "Wyoming": "WY",   
             "District of Columbia": "DC",
             "American Samoa": "AS",
             "Guam": "GU",
             "Northern Mariana Islands": "MP",
             "Puerto Rico": "PR",
             "U.S. Virgin Islands": "VI",
             "Federated States of Micronesia": "FM",
             "Marshall Islands": "MH",
             "Palau": "PW",
             "Armed Forces Americas": "AA",
             "Armed Forces Europe": "AE",
             "Armed Forces Pacific": "AP",
             "Alberta": "AB",
             "British Columbia": "BC"}
    
data_2023 = pd.read_csv('total_2023.csv')
data_2022 = pd.read_csv('total_2022.csv')
data_2021 = pd.read_csv('total_2021.csv')
data_2020 = pd.read_csv('total_2020.csv')

data_2023.drop(columns=['unitid', 'HD2023.Bureau of Economic Analysis (BEA) regions'], inplace=True)

states_to_remove = ['Armed Forces Americas', 'Armed Forces Europe', 'Armed Forces Pacific',
                    'American Samoa', 'Guam', 'Northern Marianas', 'Puerto Rico',
                    'Federated States of Micronesia', 'Palau', 'Virgin Islands',
                    'Marshall Islands']
data_2023.rename(columns={
    'HD2023.State abbreviation': 'state',
    'institution name': 'name'
}, inplace=True)
data_2023 = data_2023[~data_2023['state'].isin(states_to_remove)]
data_2023['stateAbbr'] = data_2023['state'].map(stateAbbri)

data_2022.rename(columns={
    'HD2022.State abbreviation': 'state',
    'institution name': 'name'
}, inplace=True)
data_2022 = data_2022[~data_2022['state'].isin(states_to_remove)]
data_2022['stateAbbr'] = data_2022['state'].map(stateAbbri)

data_2021.rename(columns={
    'HD2021.State abbreviation': 'state',
    'institution name': 'name'
}, inplace=True)
data_2021 = data_2021[~data_2021['state'].isin(states_to_remove)]
data_2021['stateAbbr'] = data_2021['state'].map(stateAbbri)

data_2020.rename(columns={
    'HD2020.State abbreviation': 'state',
    'institution name': 'name'
}, inplace=True)
data_2020 = data_2020[~data_2020['state'].isin(states_to_remove)]
data_2020['stateAbbr'] = data_2020['state'].map(stateAbbri)

stateRegions = {
    "Alabama": "South",
    "Alaska": "West",
    "Arizona": "West",
    "Arkansas": "South",
    "California": "West",
    "Colorado": "West",
    "Connecticut": "Northeast",
    "Delaware": "South",
    "Florida": "South",
    "Georgia": "South",
    "Hawaii": "West",
    "Idaho": "West",
    "Illinois": "Midwest",
    "Indiana": "Midwest",
    "Iowa": "Midwest",
    "Kansas": "Midwest",
    "Kentucky": "South",
    "Louisiana": "South",
    "Maine": "Northeast",
    "Maryland": "South",
    "Massachusetts": "Northeast",
    "Michigan": "Midwest",
    "Minnesota": "Midwest",
    "Mississippi": "South",
    "Missouri": "Midwest",
    "Montana": "West",
    "Nebraska": "Midwest",
    "Nevada": "West",
    "New Hampshire": "Northeast",
    "New Jersey": "Northeast",
    "New Mexico": "West",
    "New York": "Northeast",
    "North Carolina": "South",
    "North Dakota": "Midwest",
    "Ohio": "Midwest",
    "Oklahoma": "South",
    "Oregon": "West",
    "Pennsylvania": "Northeast",
    "Rhode Island": "Northeast",
    "South Carolina": "South",
    "South Dakota": "Midwest",
    "Tennessee": "South",
    "Texas": "South",
    "Utah": "West",
    "Vermont": "Northeast",
    "Virginia": "South",
    "Washington": "West",
    "West Virginia": "South",
    "Wisconsin": "Midwest",
    "Wyoming": "West",
    "District of Columbia": "South",
    "American Samoa": "West",
    "Guam": "West",
    "Northern Mariana Islands": "West",
    "Puerto Rico": "South"
}

for i in [data_2023, data_2022, data_2021, data_2020]:
    i['region'] = i['state'].map(stateRegions)

temp_state = pd.DataFrame()
temp_region = pd.DataFrame()

for year, data in zip([2020, 2021, 2022, 2023], [data_2020, data_2021, data_2022, data_2023]):
    temp_state[f'totalByState_{year}'] = data.groupby('state')['name'].count()
    temp_region[f'totalByRegion_{year}'] = data.groupby('region')['name'].count()

for year in [2024, 2025]:
    temp_state[f'totalByState_{year}'] = temp_state[f'totalByState_{year-1}']
    temp_region[f'totalByRegion_{year}'] = temp_region[f'totalByRegion_{year-1}']

temp_region

,totalByRegion_2020,totalByRegion_2021,totalByRegion_2022,totalByRegion_2023,totalByRegion_2024,totalByRegion_2025
region,,,,,,
Midwest,1322,1322,1322,1323,1323,1323
Northeast,1163,1163,1163,1163,1163,1163
South,2054,2055,2055,2055,2055,2055
West,1254,1254,1254,1253,1253,1253


In [5]:
temp_state

,totalByState_2020,totalByState_2021,totalByState_2022,totalByState_2023,totalByState_2024,totalByState_2025
state,,,,,,
Alabama,78,78,78,78,78,78
Alaska,10,10,10,11,11,11
Arizona,99,100,100,101,101,101
Arkansas,80,80,81,81,81,81
California,658,658,658,656,656,656
Colorado,86,85,85,85,85,85
Connecticut,65,65,65,65,65,65
Delaware,16,16,16,16,16,16
District of Columbia,21,21,21,21,21,21


In [6]:
closed_colleges = df.copy()

for year in [2020, 2021, 2022, 2023, 2024, 2025]:
    temp_state[f'closedByState_{year}'] = closed_colleges[closed_colleges['yearClosed'] == year].groupby('state')['name'].count()
    temp_region[f'closedByRegion_{year}'] = closed_colleges[closed_colleges['yearClosed'] == year].groupby('region')['name'].count()

temp_state.fillna(0, inplace=True)
temp_region.fillna(0, inplace=True)

temp_region = temp_region.astype(int)
temp_state = temp_state.astype(int)

temp_region

,totalByRegion_2020,totalByRegion_2021,totalByRegion_2022,totalByRegion_2023,totalByRegion_2024,totalByRegion_2025,closedByRegion_2020,closedByRegion_2021,closedByRegion_2022,closedByRegion_2023,closedByRegion_2024,closedByRegion_2025
region,,,,,,,,,,,,
Midwest,1322,1322,1322,1323,1323,1323,4,1,2,6,9,4
Northeast,1163,1163,1163,1163,1163,1163,2,2,1,4,8,0
South,2054,2055,2055,2055,2055,2055,0,4,1,1,4,2
West,1254,1254,1254,1253,1253,1253,0,2,4,2,2,0


In [7]:
for year in [2020, 2021, 2022, 2023, 2024, 2025]:
    df = df.merge(temp_state[[f'totalByState_{year}', f'closedByState_{year}']], on='state', how='left')
    df = df.merge(temp_region[[f'totalByRegion_{year}', f'closedByRegion_{year}']], on='region', how='left')

df.to_csv('full_data_bhm1.csv', index=False)
df

,name,yearEstablished,state,stateAbbreviation,locationType,locationTypeKind,typeOwned,typeProfit,level,hasUndergraduate,...,totalByRegion_2023,closedByRegion_2023,totalByState_2024,closedByState_2024,totalByRegion_2024,closedByRegion_2024,totalByState_2025,closedByState_2025,totalByRegion_2025,closedByRegion_2025
0,Alderson Broaddus University,1871,West Virginia,WV,Rural,Distant,Private,Nonprofit,4-year,True,...,2055,1,69,0,2055,4,69,0,2055,2
1,Alliance University (Formerly Nyack College),1882,New York,NY,City,Large,Private,Nonprofit,4-year,True,...,1163,4,413,3,1163,8,413,0,1163,0
2,Ancilla College,1937,Indiana,IN,Rural,Distant,Private,Nonprofit,2-year,True,...,1323,6,102,1,1323,9,102,0,1323,4
3,Becker College,1784,Massachusetts,MA,City,Midsize,Private,Nonprofit,4-year,True,...,1163,4,144,1,1163,8,144,0,1163,0
4,Bloomfield College,1868,New Jersey,NJ,Suburb,Large,Private,Nonprofit,4-year,True,...,1163,4,144,0,1163,8,144,0,1163,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,Eastern Nazarene College,1900,Massachusetts,MA,Suburb,Large,Private,Nonprofit,4-year,True,...,1163,4,144,1,1163,8,144,0,1163,0
61,Cox College,1907,Missouri,MO,City,Midsize,Private,Nonprofit,4-year,True,...,1323,6,137,0,1323,9,137,2,1323,4
62,University of Wisconsin–Platteville Richland,1967,Wisconsin,WI,Town,Distant,Public,Nonprofit,4-year,True,...,1323,6,88,4,1323,9,88,1,1323,4
63,Maryland University of Integrative Health,1974,Maryland,MD,Suburb,Large,Private,Nonprofit,graduate school,False,...,2055,1,77,0,2055,4,77,1,2055,2


## BHM 1

In [8]:
df = pd.read_csv('full_data_bhm1.csv', keep_default_na=False, na_values=[''])

years = [i for i in range(2020, 2026)]
states = df['state'].unique()
regions = df['region'].unique()

print(f"Years: {years}\nStates: {states}\nRegions: {regions}")
df

Years: [2020, 2021, 2022, 2023, 2024, 2025]
States: ['West Virginia' 'New York' 'Indiana' 'Massachusetts' 'New Jersey' 'Ohio'
 'Pennsylvania' 'Wisconsin' 'Michigan' 'Missouri' 'Florida' 'California'
 'Utah' 'Iowa' 'Alabama' 'Illinois' 'New Hampshire' 'Vermont' 'Tennessee'
 'Oregon' 'Nebraska' 'South Dakota' 'Nevada' 'Virginia' 'Delaware'
 'Oklahoma' 'Maryland' 'Texas']
Regions: ['South' 'Northeast' 'Midwest' 'West']


,name,yearEstablished,state,stateAbbreviation,locationType,locationTypeKind,typeOwned,typeProfit,level,hasUndergraduate,...,totalByRegion_2023,closedByRegion_2023,totalByState_2024,closedByState_2024,totalByRegion_2024,closedByRegion_2024,totalByState_2025,closedByState_2025,totalByRegion_2025,closedByRegion_2025
0,Alderson Broaddus University,1871,West Virginia,WV,Rural,Distant,Private,Nonprofit,4-year,True,...,2055,1,69,0,2055,4,69,0,2055,2
1,Alliance University (Formerly Nyack College),1882,New York,NY,City,Large,Private,Nonprofit,4-year,True,...,1163,4,413,3,1163,8,413,0,1163,0
2,Ancilla College,1937,Indiana,IN,Rural,Distant,Private,Nonprofit,2-year,True,...,1323,6,102,1,1323,9,102,0,1323,4
3,Becker College,1784,Massachusetts,MA,City,Midsize,Private,Nonprofit,4-year,True,...,1163,4,144,1,1163,8,144,0,1163,0
4,Bloomfield College,1868,New Jersey,NJ,Suburb,Large,Private,Nonprofit,4-year,True,...,1163,4,144,0,1163,8,144,0,1163,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,Eastern Nazarene College,1900,Massachusetts,MA,Suburb,Large,Private,Nonprofit,4-year,True,...,1163,4,144,1,1163,8,144,0,1163,0
61,Cox College,1907,Missouri,MO,City,Midsize,Private,Nonprofit,4-year,True,...,1323,6,137,0,1323,9,137,2,1323,4
62,University of Wisconsin–Platteville Richland,1967,Wisconsin,WI,Town,Distant,Public,Nonprofit,4-year,True,...,1323,6,88,4,1323,9,88,1,1323,4
63,Maryland University of Integrative Health,1974,Maryland,MD,Suburb,Large,Private,Nonprofit,graduate school,False,...,2055,1,77,0,2055,4,77,1,2055,2


In [9]:
state_data = []
region_data = []
for year in years:
    for state in states:
        state_df = df[df['state'] == state]
        total = state_df[f'totalByState_{year}'].sum()
        closed = state_df[f'closedByState_{year}'].sum()
        if total > 0:
            state_data.append({'state': state, 'year': year, 'total': total, 'closed': closed})
    for region in regions:
        region_df = df[df['region'] == region]
        total = region_df[f'totalByRegion_{year}'].sum()
        closed = region_df[f'closedByRegion_{year}'].sum()
        if total > 0:
            region_data.append({'region': region, 'year': year, 'total': total, 'closed': closed})

state_df = pd.DataFrame(state_data)
region_df = pd.DataFrame(region_data)

state_df

,state,year,total,closed
0,West Virginia,2020,138,0
1,New York,2020,2891,0
2,Indiana,2020,204,0
3,Massachusetts,2020,432,3
4,New Jersey,2020,144,0
...,...,...,...,...
163,Virginia,2025,138,0
164,Delaware,2025,16,0
165,Oklahoma,2025,93,0
166,Maryland,2025,77,1


In [10]:
with pm.Model() as model:
    mu_global = pm.Normal('mu_global', mu=0, sigma=10)
    sigma_region = pm.HalfNormal('sigma_region', sigma=5)
    region_effects = pm.Normal('region_effects', mu=0, sigma=sigma_region, shape=len(regions))
    
    region_idx_map = {region: i for i, region in enumerate(regions)}

    state_to_region = dict(zip(df['state'], df['region']))
    state_region_idx = [region_idx_map[state_to_region[state_df.loc[i, 'state']]] for i in range(len(state_df))]
    
    sigma_state = pm.HalfNormal('sigma_state', sigma=5)
    state_effects = pm.Normal('state_effects', mu=region_effects[state_region_idx], sigma=sigma_state, shape=len(state_df))
    
    logit_p = mu_global + state_effects
    p = pm.Deterministic('p', pm.math.sigmoid(logit_p))
    y = pm.Binomial('y', n=state_df['total'], p=p, observed=state_df['closed'])
    
    trace = pm.sample(return_inferencedata=True, random_seed=42)

az.to_netcdf(trace, 'bhm/trace_bhm_1.nc')
az.summary(trace)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu_global, sigma_region, region_effects, sigma_state, state_effects]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 4 seconds.
There were 83 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
mu_global,-7.163,0.566,-8.249,-6.079,0.072,0.085,94.0,51.0,1.05
region_effects[0],-0.181,0.555,-1.529,0.696,0.056,0.074,154.0,58.0,1.03
region_effects[1],0.006,0.548,-1.162,1.039,0.060,0.084,131.0,60.0,1.04
region_effects[2],0.275,0.572,-0.743,1.251,0.068,0.107,148.0,52.0,1.03
region_effects[3],-0.016,0.576,-1.223,0.973,0.060,0.084,158.0,65.0,1.03
...,...,...,...,...,...,...,...,...,...
p[163],0.001,0.002,0.000,0.004,0.000,0.000,2961.0,2347.0,1.00
p[164],0.002,0.006,0.000,0.009,0.000,0.000,2809.0,2507.0,1.00
p[165],0.001,0.002,0.000,0.005,0.000,0.000,3085.0,2085.0,1.00
p[166],0.007,0.008,0.000,0.020,0.000,0.000,4266.0,2690.0,1.00


In [11]:
with pm.Model() as model:
    mu_global = pm.Normal('mu_global', mu=0, sigma=10)
    sigma_region = pm.HalfNormal('sigma_region', sigma=5)
    region_effects = pm.Normal('region_effects', mu=0, sigma=sigma_region, shape=len(regions))
    
    region_idx_map = {region: i for i, region in enumerate(regions)}

    state_to_region = dict(zip(df['state'], df['region']))
    state_region_idx = [region_idx_map[state_to_region[state_df.loc[i, 'state']]] for i in range(len(state_df))]
    
    sigma_state = pm.HalfNormal('sigma_state', sigma=5)
    state_effects = pm.Normal('state_effects', mu=region_effects[state_region_idx], sigma=sigma_state, shape=len(state_df))
    
    logit_p = mu_global + state_effects
    p = pm.Deterministic('p', pm.math.sigmoid(logit_p))
    y = pm.Binomial('y', n=state_df['total'], p=p, observed=state_df['closed'])

    trace = pm.sample(4000, tune=2000, target_accept=0.9, return_inferencedata=True, random_seed=42)

az.to_netcdf(trace, 'bhm/trace_bhm_2.nc')
az.summary(trace)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu_global, sigma_region, region_effects, sigma_state, state_effects]


Output()

Sampling 4 chains for 2_000 tune and 4_000 draw iterations (8_000 + 16_000 draws total) took 8 seconds.
There were 285 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
mu_global,-7.153,0.464,-8.035,-6.307,0.019,0.017,619.0,890.0,1.01
region_effects[0],-0.204,0.454,-1.183,0.582,0.013,0.016,1453.0,1491.0,1.00
region_effects[1],-0.010,0.440,-0.909,0.876,0.012,0.016,1654.0,1597.0,1.00
region_effects[2],0.241,0.446,-0.471,1.221,0.013,0.016,1239.0,1626.0,1.01
region_effects[3],-0.024,0.461,-0.990,0.886,0.012,0.015,2103.0,1851.0,1.00
...,...,...,...,...,...,...,...,...,...
p[163],0.001,0.002,0.000,0.004,0.000,0.000,15906.0,8878.0,1.00
p[164],0.002,0.006,0.000,0.009,0.000,0.000,16109.0,10064.0,1.00
p[165],0.001,0.002,0.000,0.005,0.000,0.000,10596.0,7255.0,1.00
p[166],0.007,0.008,0.000,0.020,0.000,0.000,20415.0,10540.0,1.00
